In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

In [2]:
#Load Dataset
train_df = pd.read_csv(
    "../data/train_FD001.txt",
    sep=r"\s+",
    header=None
)

In [3]:
column_names = (
    ["engine_id", "cycle"]
    + [f"op_setting_{i}" for i in range(1, 4)]
    + [f"sensor_{i}" for i in range(1, 22)]
)

train_df.columns = column_names

In [4]:
df = train_df.copy()

In [5]:
#Create RUL
max_cycles = df.groupby("engine_id")["cycle"].max()

df["RUL"] = (
    df["engine_id"].map(max_cycles)
    - df["cycle"]
)

In [6]:
#Create Classification Labels
df["health_status"] = pd.cut(
    df["RUL"],
    bins=[-1, 30, 100, float("inf")],
    labels=["Critical", "Warning", "Healthy"]
)

In [7]:
#Drop useless features
columns_to_drop = [
    "engine_id",
    "op_setting_3",
    "sensor_1",
    "sensor_5",
    "sensor_10",
    "sensor_16",
    "sensor_18",
    "sensor_19",
    "RUL"
]

df = df.drop(columns=columns_to_drop)

In [8]:
#Create X and y
X = df.drop("health_status", axis=1)

y = df["health_status"]

In [9]:
X.shape

(20631, 18)

In [10]:
X.head()

,cycle,op_setting_1,op_setting_2,sensor_2,sensor_3,sensor_4,sensor_6,sensor_7,sensor_8,sensor_9,sensor_11,sensor_12,sensor_13,sensor_14,sensor_15,sensor_17,sensor_20,sensor_21
0,1,-0.0007,-0.0004,641.82,1589.70,1400.60,21.61,554.36,2388.06,9046.19,47.47,521.66,2388.02,8138.62,8.4195,392,39.06,23.4190
1,2,0.0019,-0.0003,642.15,1591.82,1403.14,21.61,553.75,2388.04,9044.07,47.49,522.28,2388.07,8131.49,8.4318,392,39.00,23.4236
2,3,-0.0043,0.0003,642.35,1587.99,1404.20,21.61,554.26,2388.08,9052.94,47.27,522.42,2388.03,8133.23,8.4178,390,38.95,23.3442
3,4,0.0007,0.0000,642.35,1582.79,1401.87,21.61,554.45,2388.11,9049.48,47.13,522.86,2388.08,8133.83,8.3682,392,38.88,23.3739
4,5,-0.0019,-0.0002,642.37,1582.85,1406.22,21.61,554.00,2388.06,9055.15,47.28,522.19,2388.04,8133.80,8.4294,393,38.90,23.4044


In [11]:
y.value_counts()

health_status
Healthy     10531
Warning      7000
Critical     3100
Name: count, dtype: int64

In [12]:
#Train test split
from sklearn.model_selection import train_test_split

In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y #While splitting the data, keep the same class proportions in train and test sets as in the original dataset.
)

In [14]:
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (16504, 18)
X_test : (4127, 18)
y_train: (16504,)
y_test : (4127,)


In [15]:
print(y_train.value_counts(normalize=True) * 100)

health_status
Healthy     51.042172
Warning     33.931168
Critical    15.026660
Name: proportion, dtype: float64


In [16]:
#Train random forest
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

In [17]:
rf = RandomForestClassifier(
    n_estimators=100, #ie. build 100 decision trees
    random_state=42
)

rf.fit(X_train, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [18]:
y_pred = rf.predict(X_test)

In [19]:
accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)

Accuracy: 0.8332929488732735


In [20]:
#Confusion Matrix
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

cm

array([[ 527,    0,   93],
       [   0, 1880,  227],
       [  56,  312, 1032]])

In [21]:
#Classification Report
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

    Critical       0.90      0.85      0.88       620
     Healthy       0.86      0.89      0.87      2107
     Warning       0.76      0.74      0.75      1400

    accuracy                           0.83      4127
   macro avg       0.84      0.83      0.83      4127
weighted avg       0.83      0.83      0.83      4127



In [22]:
#Feature Importance (tells which sensors the model actually uses to determine Healthy / Warning / Critical.)
import pandas as pd

feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf.feature_importances_
})

feature_importance.sort_values(
    by="Importance",
    ascending=False
).head(15)

,Feature,Importance
0,cycle,0.211133
10,sensor_11,0.116132
5,sensor_4,0.100004
9,sensor_9,0.068055
11,sensor_12,0.066762
7,sensor_7,0.062060
14,sensor_15,0.057294
13,sensor_14,0.056421
17,sensor_21,0.050277
3,sensor_2,0.040159


In [23]:
#Try to improve accuracy beyond 83%
from sklearn.ensemble import RandomForestClassifier

rf2 = RandomForestClassifier(
    n_estimators=300,
    max_depth=20,
    min_samples_split=5,
    random_state=42,
    n_jobs=-1
)

rf2.fit(X_train, y_train)

y_pred2 = rf2.predict(X_test)

from sklearn.metrics import accuracy_score

print(
    "Accuracy:",
    accuracy_score(y_test, y_pred2)
)

Accuracy: 0.8345044826750666


In [24]:
#Lowest importance feature
feature_importance.sort_values(
    by="Importance",
    ascending=True
).head(10)

,Feature,Importance
6,sensor_6,0.000365
2,op_setting_2,0.015777
15,sensor_17,0.017325
1,op_setting_1,0.022963
8,sensor_8,0.023891
12,sensor_13,0.024032
4,sensor_3,0.032930
16,sensor_20,0.034421
3,sensor_2,0.040159
17,sensor_21,0.050277


In [25]:
#Create a reduced feature set
drop_cols = [
    "sensor_6",
    "op_setting_2",
    "sensor_17"
]

X_reduced = X.drop(columns=drop_cols)

X_reduced.shape

(20631, 15)

In [26]:
#Train test split again
from sklearn.model_selection import train_test_split

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_reduced,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [27]:
rf_reduced = RandomForestClassifier(
    n_estimators=300,
    max_depth=20,
    min_samples_split=5,
    random_state=42,
    n_jobs=-1
)

rf_reduced.fit(X_train_r, y_train_r)

y_pred_r = rf_reduced.predict(X_test_r)

from sklearn.metrics import accuracy_score

print(
    "Reduced Accuracy:",
    accuracy_score(y_test_r, y_pred_r)
)

Reduced Accuracy: 0.8369275502786527


In [28]:
from sklearn.metrics import classification_report

print(classification_report(y_test_r, y_pred_r))

              precision    recall  f1-score   support

    Critical       0.91      0.85      0.88       620
     Healthy       0.86      0.89      0.88      2107
     Warning       0.77      0.75      0.76      1400

    accuracy                           0.84      4127
   macro avg       0.84      0.83      0.84      4127
weighted avg       0.84      0.84      0.84      4127



In [29]:
from sklearn.metrics import confusion_matrix

confusion_matrix(y_test_r, y_pred_r)

array([[ 527,    0,   93],
       [   0, 1878,  229],
       [  55,  296, 1049]])

In [30]:
#Save the model
import joblib

joblib.dump(
    rf_reduced,
    "../models/rf_health_classifier.pkl"
)

['../models/rf_health_classifier.pkl']

In [31]:
#Also save feature list (during dashboard prediction, we'll need the exact same feature order.)
joblib.dump(
    X_reduced.columns.tolist(),
    "../models/model_features.pkl"
)

['../models/model_features.pkl']

In [32]:
#Save label names
joblib.dump(
    ["Critical", "Warning", "Healthy"],
    "../models/class_labels.pkl"
)

['../models/class_labels.pkl']

In [33]:
#----------------------------------MODEL COMPARISON-------------------------------

In [35]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

lr = LogisticRegression(max_iter=1000)

lr.fit(X_train_r, y_train_r)

lr_pred = lr.predict(X_test_r)

print(
    "Logistic Accuracy:",
    accuracy_score(y_test_r, lr_pred)
)

Logistic Accuracy: 0.7940392536951781


C:\Users\saees\Miniconda3\lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [36]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

lr_scaled = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=5000))
])

lr_scaled.fit(X_train_r, y_train_r)

pred = lr_scaled.predict(X_test_r)

print(
    "Scaled Logistic Accuracy:",
    accuracy_score(y_test_r, pred)
)

Scaled Logistic Accuracy: 0.820450690574267


In [37]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

dt = DecisionTreeClassifier(
    max_depth=20,
    random_state=42
)

dt.fit(X_train_r, y_train_r)

dt_pred = dt.predict(X_test_r)

print(
    "Decision Tree Accuracy:",
    accuracy_score(y_test_r, dt_pred)
)

Decision Tree Accuracy: 0.7685970438575236


In [41]:
!pip install xgboost

  Using cached xgboost-3.2.0-py3-none-win_amd64.whl.metadata (2.1 kB)
Using cached xgboost-3.2.0-py3-none-win_amd64.whl (101.7 MB)


In [43]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y_train_enc = le.fit_transform(y_train_r)
y_test_enc = le.transform(y_test_r)

print(le.classes_)

['Critical' 'Healthy' 'Warning']


In [44]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score

xgb = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    random_state=42
)

xgb.fit(X_train_r, y_train_enc)

xgb_pred = xgb.predict(X_test_r)

print(
    "XGBoost Accuracy:",
    accuracy_score(y_test_enc, xgb_pred)
)

XGBoost Accuracy: 0.831112188030046


In [45]:
pred_labels = le.inverse_transform(xgb_pred)

print(pred_labels[:10])

['Warning' 'Healthy' 'Warning' 'Healthy' 'Healthy' 'Healthy' 'Warning'
 'Healthy' 'Warning' 'Warning']


In [49]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score

xgb2 = XGBClassifier(
    n_estimators=500,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

xgb2.fit(X_train_r, y_train_enc)

xgb2_pred = xgb2.predict(X_test_r)

print(
    "XGBoost2 Accuracy:",
    accuracy_score(y_test_enc, xgb2_pred)
)

XGBoost2 Accuracy: 0.8340198691543494


In [47]:
#Comparison Table
results = {
    "Model": [
        "Decision Tree",
        "Logistic Regression",
        "Scaled Logistic Regression",
        "Random Forest",
        "XGBoost"
    ],
    "Accuracy": [
        0.7686,
        0.7940,
        0.8205,
        0.8369,
        0.8311
    ]
}

pd.DataFrame(results).sort_values(
    by="Accuracy",
    ascending=False
)

,Model,Accuracy
3,Random Forest,0.8369
4,XGBoost,0.8311
2,Scaled Logistic Regression,0.8205
1,Logistic Regression,0.7940
0,Decision Tree,0.7686


In [51]:
df.columns

Index(['cycle', 'op_setting_1', 'op_setting_2', 'sensor_2', 'sensor_3',
       'sensor_4', 'sensor_6', 'sensor_7', 'sensor_8', 'sensor_9', 'sensor_11',
       'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15', 'sensor_17',
       'sensor_20', 'sensor_21', 'health_status'],
      dtype='object')

In [52]:
train_df.columns

Index(['engine_id', 'cycle', 'op_setting_1', 'op_setting_2', 'op_setting_3',
       'sensor_1', 'sensor_2', 'sensor_3', 'sensor_4', 'sensor_5', 'sensor_6',
       'sensor_7', 'sensor_8', 'sensor_9', 'sensor_10', 'sensor_11',
       'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15', 'sensor_16',
       'sensor_17', 'sensor_18', 'sensor_19', 'sensor_20', 'sensor_21'],
      dtype='object')

In [53]:
train_df.head()

,engine_id,cycle,op_setting_1,op_setting_2,op_setting_3,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,...,sensor_12,sensor_13,sensor_14,sensor_15,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21
0,1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,...,521.66,2388.02,8138.62,8.4195,0.03,392,2388,100.0,39.06,23.4190
1,1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,...,522.28,2388.07,8131.49,8.4318,0.03,392,2388,100.0,39.00,23.4236
2,1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,...,522.42,2388.03,8133.23,8.4178,0.03,390,2388,100.0,38.95,23.3442
3,1,4,0.0007,0.0000,100.0,518.67,642.35,1582.79,1401.87,14.62,...,522.86,2388.08,8133.83,8.3682,0.03,392,2388,100.0,38.88,23.3739
4,1,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,...,522.19,2388.04,8133.80,8.4294,0.03,393,2388,100.0,38.90,23.4044


In [54]:
train_df.shape

(20631, 26)

In [55]:
"RUL" in train_df.columns

False

In [56]:
df2 = train_df.copy()

In [57]:
max_cycles = df2.groupby("engine_id")["cycle"].max()

df2["RUL"] = (
    df2["engine_id"].map(max_cycles)
    - df2["cycle"]
)

In [58]:
df2[["engine_id", "cycle", "RUL"]].head()

,engine_id,cycle,RUL
0,1,1,191
1,1,2,190
2,1,3,189
3,1,4,188
4,1,5,187


In [59]:
df2["max_cycle"] = df2["engine_id"].map(max_cycles)

df2["life_progress"] = (
    df2["cycle"] / df2["max_cycle"]
)

In [60]:
df2[
    ["engine_id","cycle","max_cycle","life_progress"]
].head()

,engine_id,cycle,max_cycle,life_progress
0,1,1,192,0.005208
1,1,2,192,0.010417
2,1,3,192,0.015625
3,1,4,192,0.020833
4,1,5,192,0.026042


In [61]:
#Create health lables again
df2["health_status"] = pd.cut(
    df2["RUL"],
    bins=[-1, 30, 100, float("inf")],
    labels=["Critical", "Warning", "Healthy"]
)

In [62]:
df2["health_status"].value_counts()

health_status
Healthy     10531
Warning      7000
Critical     3100
Name: count, dtype: int64

In [63]:
df2.columns

Index(['engine_id', 'cycle', 'op_setting_1', 'op_setting_2', 'op_setting_3',
       'sensor_1', 'sensor_2', 'sensor_3', 'sensor_4', 'sensor_5', 'sensor_6',
       'sensor_7', 'sensor_8', 'sensor_9', 'sensor_10', 'sensor_11',
       'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15', 'sensor_16',
       'sensor_17', 'sensor_18', 'sensor_19', 'sensor_20', 'sensor_21', 'RUL',
       'max_cycle', 'life_progress', 'health_status'],
      dtype='object')

In [64]:
features = [
    "cycle",
    "life_progress",
    "op_setting_1",
    "op_setting_2",
    "sensor_2",
    "sensor_3",
    "sensor_4",
    "sensor_6",
    "sensor_7",
    "sensor_8",
    "sensor_9",
    "sensor_11",
    "sensor_12",
    "sensor_13",
    "sensor_14",
    "sensor_15",
    "sensor_17",
    "sensor_20",
    "sensor_21"
]

X = df2[features]
y = df2["health_status"]

In [65]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [66]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=20,
    min_samples_split=5,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

pred = rf.predict(X_test)

print(
    "Accuracy:",
    accuracy_score(y_test, pred)
)

Accuracy: 0.9561424763750909


In [67]:
import pandas as pd

feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf.feature_importances_
})

feature_importance.sort_values(
    by="Importance",
    ascending=False
).head(10)

,Feature,Importance
1,life_progress,0.396802
0,cycle,0.176756
11,sensor_11,0.074318
6,sensor_4,0.057995
12,sensor_12,0.052062
8,sensor_7,0.037094
15,sensor_15,0.035119
10,sensor_9,0.031964
14,sensor_14,0.026989
18,sensor_21,0.020901


In [68]:
feature_importance.sort_values(
    by="Importance",
    ascending=False
)

,Feature,Importance
1,life_progress,0.396802
0,cycle,0.176756
11,sensor_11,0.074318
6,sensor_4,0.057995
12,sensor_12,0.052062
8,sensor_7,0.037094
15,sensor_15,0.035119
10,sensor_9,0.031964
14,sensor_14,0.026989
18,sensor_21,0.020901


In [69]:
###DATA LEAKAGE HAPPENED DUE TO life_progress, max_cycles SO WE GOT 95% ----BAD RESET EVERYTHING

In [111]:
df2 = df2.drop(
    columns=["max_cycle", "life_progress"],
    errors="ignore"
)

In [112]:
df2.columns

Index(['engine_id', 'cycle', 'op_setting_1', 'op_setting_2', 'op_setting_3',
       'sensor_1', 'sensor_2', 'sensor_3', 'sensor_4', 'sensor_5', 'sensor_6',
       'sensor_7', 'sensor_8', 'sensor_9', 'sensor_10', 'sensor_11',
       'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15', 'sensor_16',
       'sensor_17', 'sensor_18', 'sensor_19', 'sensor_20', 'sensor_21', 'RUL',
       'health_status', 'sensor_11_avg5', 'sensor_4_avg5', 'sensor_15_avg5',
       'sensor_2_avg5', 'sensor_17_avg5'],
      dtype='object')

In [113]:
base_features = [
    "cycle",
    "op_setting_1",
    "op_setting_2",
    "sensor_2",
    "sensor_3",
    "sensor_4",
    "sensor_6",
    "sensor_7",
    "sensor_8",
    "sensor_9",
    "sensor_11",
    "sensor_12",
    "sensor_13",
    "sensor_14",
    "sensor_15",
    "sensor_17",
    "sensor_20",
    "sensor_21"
]

In [114]:
X = df2[base_features]
y = df2["health_status"]

In [115]:
print("life_progress" in df2.columns)
print("max_cycle" in df2.columns)

False
False


In [116]:
#Create sensor trend features
important_sensors = [
    "sensor_11",
    "sensor_4",
    "sensor_15",
    "sensor_2",
    "sensor_17"
]

for sensor in important_sensors:
    df2[f"{sensor}_diff"] = (
        df2.groupby("engine_id")[sensor]
           .diff()
           .fillna(0)
    )

In [117]:
#Create rolling averages
for sensor in important_sensors:
    df2[f"{sensor}_avg5"] = (
        df2.groupby("engine_id")[sensor]
           .transform(
               lambda x: x.rolling(
                   window=5,
                   min_periods=1
               ).mean()
           )
    )

In [118]:
#Build feature list
base_features = [
    "cycle",
    "op_setting_1",
    "op_setting_2",
    "sensor_2",
    "sensor_3",
    "sensor_4",
    "sensor_6",
    "sensor_7",
    "sensor_8",
    "sensor_9",
    "sensor_11",
    "sensor_12",
    "sensor_13",
    "sensor_14",
    "sensor_15",
    "sensor_17",
    "sensor_20",
    "sensor_21"
]

new_features = (
    [f"{s}_diff" for s in important_sensors]
    +
    [f"{s}_avg5" for s in important_sensors]
)

features = base_features + new_features

In [119]:
X = df2[features]
y = df2["health_status"]

In [120]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [121]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=20,
    min_samples_split=5,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

pred = rf.predict(X_test)

print(
    "Accuracy:",
    accuracy_score(y_test, pred)
)

Accuracy: 0.851223649139811


In [81]:
feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf.feature_importances_
})

feature_importance.sort_values(
    by="Importance",
    ascending=False
).head(20)

,Feature,Importance
0,cycle,0.179216
23,sensor_11_avg5,0.089803
24,sensor_4_avg5,0.089052
25,sensor_15_avg5,0.086274
26,sensor_2_avg5,0.062172
9,sensor_9,0.057267
27,sensor_17_avg5,0.053445
13,sensor_14,0.049400
10,sensor_11,0.040823
5,sensor_4,0.033380


In [83]:
#Now try rolling std deviations
for sensor in important_sensors:
    df2[f"{sensor}_std5"] = (
        df2.groupby("engine_id")[sensor]
           .transform(
               lambda x: x.rolling(
                   window=5,
                   min_periods=1
               ).std()
           )
           .fillna(0)
    )

In [84]:
std_features = [
    f"{s}_std5"
    for s in important_sensors
]

features = (
    base_features
    + new_features
    + std_features
)

In [85]:
X = df2[features]
y = df2["health_status"]

In [86]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [87]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=20,
    min_samples_split=5,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

pred = rf.predict(X_test)

print(
    "Accuracy:",
    accuracy_score(y_test, pred)
)

Accuracy: 0.8497698085776593


In [91]:
[col for col in df2.columns if "std" in col or "diff" in col]

['sensor_11_diff',
 'sensor_4_diff',
 'sensor_15_diff',
 'sensor_2_diff',
 'sensor_17_diff',
 'sensor_11_std5',
 'sensor_4_std5',
 'sensor_15_std5',
 'sensor_2_std5',
 'sensor_17_std5']

In [122]:
df2.columns

Index(['engine_id', 'cycle', 'op_setting_1', 'op_setting_2', 'op_setting_3',
       'sensor_1', 'sensor_2', 'sensor_3', 'sensor_4', 'sensor_5', 'sensor_6',
       'sensor_7', 'sensor_8', 'sensor_9', 'sensor_10', 'sensor_11',
       'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15', 'sensor_16',
       'sensor_17', 'sensor_18', 'sensor_19', 'sensor_20', 'sensor_21', 'RUL',
       'health_status', 'sensor_11_avg5', 'sensor_4_avg5', 'sensor_15_avg5',
       'sensor_2_avg5', 'sensor_17_avg5', 'sensor_11_diff', 'sensor_4_diff',
       'sensor_15_diff', 'sensor_2_diff', 'sensor_17_diff'],
      dtype='object')

In [123]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=20,
    min_samples_split=5,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

pred = rf.predict(X_test)

print(
    "Accuracy:",
    accuracy_score(y_test, pred)
)

Accuracy: 0.851223649139811


In [124]:
from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(y_test, pred))

confusion_matrix(y_test, pred)

              precision    recall  f1-score   support

    Critical       0.92      0.87      0.90       620
     Healthy       0.87      0.90      0.89      2107
     Warning       0.79      0.77      0.78      1400

    accuracy                           0.85      4127
   macro avg       0.86      0.85      0.85      4127
weighted avg       0.85      0.85      0.85      4127



array([[ 538,    0,   82],
       [   0, 1904,  203],
       [  44,  285, 1071]])

In [125]:
feature_importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": rf.feature_importances_
})

feature_importance.sort_values(
    by="Importance",
    ascending=False
).head(20)

,Feature,Importance
0,cycle,0.179216
23,sensor_11_avg5,0.089803
24,sensor_4_avg5,0.089052
25,sensor_15_avg5,0.086274
26,sensor_2_avg5,0.062172
9,sensor_9,0.057267
27,sensor_17_avg5,0.053445
13,sensor_14,0.049400
10,sensor_11,0.040823
5,sensor_4,0.033380


In [126]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score

gb = GradientBoostingClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    random_state=42
)

gb.fit(X_train, y_train)

gb_pred = gb.predict(X_test)

print(
    "Gradient Boost Accuracy:",
    accuracy_score(y_test, gb_pred)
)

Gradient Boost Accuracy: 0.8495275018173007


In [127]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

In [128]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score

xgb = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    random_state=42
)

xgb.fit(X_train, y_train_enc)

xgb_pred = xgb.predict(X_test)

print(
    "XGBoost Accuracy:",
    accuracy_score(y_test_enc, xgb_pred)
)

XGBoost Accuracy: 0.8521928761812455


In [130]:
print("RUL" in X_train.columns)

False


In [131]:
X_train.columns

Index(['cycle', 'op_setting_1', 'op_setting_2', 'sensor_2', 'sensor_3',
       'sensor_4', 'sensor_6', 'sensor_7', 'sensor_8', 'sensor_9', 'sensor_11',
       'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15', 'sensor_17',
       'sensor_20', 'sensor_21', 'sensor_11_diff', 'sensor_4_diff',
       'sensor_15_diff', 'sensor_2_diff', 'sensor_17_diff', 'sensor_11_avg5',
       'sensor_4_avg5', 'sensor_15_avg5', 'sensor_2_avg5', 'sensor_17_avg5'],
      dtype='object')

In [132]:
df2.columns

Index(['engine_id', 'cycle', 'op_setting_1', 'op_setting_2', 'op_setting_3',
       'sensor_1', 'sensor_2', 'sensor_3', 'sensor_4', 'sensor_5', 'sensor_6',
       'sensor_7', 'sensor_8', 'sensor_9', 'sensor_10', 'sensor_11',
       'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15', 'sensor_16',
       'sensor_17', 'sensor_18', 'sensor_19', 'sensor_20', 'sensor_21', 'RUL',
       'health_status', 'sensor_11_avg5', 'sensor_4_avg5', 'sensor_15_avg5',
       'sensor_2_avg5', 'sensor_17_avg5', 'sensor_11_diff', 'sensor_4_diff',
       'sensor_15_diff', 'sensor_2_diff', 'sensor_17_diff'],
      dtype='object')

In [133]:
important_sensors = [
    "sensor_11",
    "sensor_4",
    "sensor_15",
    "sensor_2",
    "sensor_17"
]

for sensor in important_sensors:
    df2[f"{sensor}_avg10"] = (
        df2.groupby("engine_id")[sensor]
        .transform(
            lambda x: x.rolling(
                10,
                min_periods=1
            ).mean()
        )
    )

In [134]:
X = df2.drop(
    ["health_status"],
    axis=1
)

X = X.drop(
    ["engine_id", "RUL"],
    axis=1
)

y = df2["health_status"]

In [135]:
X = df2.drop(
    ["health_status"],
    axis=1
)

X = X.drop(
    ["engine_id", "RUL"],
    axis=1
)

y = df2["health_status"]

In [136]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=20,
    min_samples_split=5,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

pred = rf.predict(X_test)

print(
    "Accuracy:",
    accuracy_score(y_test, pred)
)

Accuracy: 0.851223649139811


In [137]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test,
        pred
    )
)

              precision    recall  f1-score   support

    Critical       0.92      0.87      0.90       620
     Healthy       0.87      0.90      0.89      2107
     Warning       0.79      0.77      0.78      1400

    accuracy                           0.85      4127
   macro avg       0.86      0.85      0.85      4127
weighted avg       0.85      0.85      0.85      4127



In [138]:
import pandas as pd

feature_importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": rf.feature_importances_
})

feature_importance.sort_values(
    by="Importance",
    ascending=False
).head(20)

,Feature,Importance
0,cycle,0.179216
23,sensor_11_avg5,0.089803
24,sensor_4_avg5,0.089052
25,sensor_15_avg5,0.086274
26,sensor_2_avg5,0.062172
9,sensor_9,0.057267
27,sensor_17_avg5,0.053445
13,sensor_14,0.049400
10,sensor_11,0.040823
5,sensor_4,0.033380


In [139]:
[col for col in df2.columns if "avg10" in col]

['sensor_11_avg10',
 'sensor_4_avg10',
 'sensor_15_avg10',
 'sensor_2_avg10',
 'sensor_17_avg10']

In [140]:
X_train.columns

Index(['cycle', 'op_setting_1', 'op_setting_2', 'sensor_2', 'sensor_3',
       'sensor_4', 'sensor_6', 'sensor_7', 'sensor_8', 'sensor_9', 'sensor_11',
       'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15', 'sensor_17',
       'sensor_20', 'sensor_21', 'sensor_11_diff', 'sensor_4_diff',
       'sensor_15_diff', 'sensor_2_diff', 'sensor_17_diff', 'sensor_11_avg5',
       'sensor_4_avg5', 'sensor_15_avg5', 'sensor_2_avg5', 'sensor_17_avg5'],
      dtype='object')

In [141]:
df2.columns

Index(['engine_id', 'cycle', 'op_setting_1', 'op_setting_2', 'op_setting_3',
       'sensor_1', 'sensor_2', 'sensor_3', 'sensor_4', 'sensor_5', 'sensor_6',
       'sensor_7', 'sensor_8', 'sensor_9', 'sensor_10', 'sensor_11',
       'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15', 'sensor_16',
       'sensor_17', 'sensor_18', 'sensor_19', 'sensor_20', 'sensor_21', 'RUL',
       'health_status', 'sensor_11_avg5', 'sensor_4_avg5', 'sensor_15_avg5',
       'sensor_2_avg5', 'sensor_17_avg5', 'sensor_11_diff', 'sensor_4_diff',
       'sensor_15_diff', 'sensor_2_diff', 'sensor_17_diff', 'sensor_11_avg10',
       'sensor_4_avg10', 'sensor_15_avg10', 'sensor_2_avg10',
       'sensor_17_avg10'],
      dtype='object')

In [142]:
len(X_train.columns)

28

In [143]:
X_train.columns.tolist()

['cycle',
 'op_setting_1',
 'op_setting_2',
 'sensor_2',
 'sensor_3',
 'sensor_4',
 'sensor_6',
 'sensor_7',
 'sensor_8',
 'sensor_9',
 'sensor_11',
 'sensor_12',
 'sensor_13',
 'sensor_14',
 'sensor_15',
 'sensor_17',
 'sensor_20',
 'sensor_21',
 'sensor_11_diff',
 'sensor_4_diff',
 'sensor_15_diff',
 'sensor_2_diff',
 'sensor_17_diff',
 'sensor_11_avg5',
 'sensor_4_avg5',
 'sensor_15_avg5',
 'sensor_2_avg5',
 'sensor_17_avg5']

In [144]:
X = df2.drop(
    ["health_status"],
    axis=1
)

X = X.drop(
    ["engine_id", "RUL"],
    axis=1
)

y = df2["health_status"]

In [145]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [146]:
len(X_train.columns)

40

In [147]:
sorted(set(X_train.columns) - set([
 'cycle',
 'op_setting_1',
 'op_setting_2',
 'sensor_2',
 'sensor_3',
 'sensor_4',
 'sensor_6',
 'sensor_7',
 'sensor_8',
 'sensor_9',
 'sensor_11',
 'sensor_12',
 'sensor_13',
 'sensor_14',
 'sensor_15',
 'sensor_17',
 'sensor_20',
 'sensor_21',
 'sensor_11_diff',
 'sensor_4_diff',
 'sensor_15_diff',
 'sensor_2_diff',
 'sensor_17_diff',
 'sensor_11_avg5',
 'sensor_4_avg5',
 'sensor_15_avg5',
 'sensor_2_avg5',
 'sensor_17_avg5'
]))

['op_setting_3',
 'sensor_1',
 'sensor_10',
 'sensor_11_avg10',
 'sensor_15_avg10',
 'sensor_16',
 'sensor_17_avg10',
 'sensor_18',
 'sensor_19',
 'sensor_2_avg10',
 'sensor_4_avg10',
 'sensor_5']

In [148]:
rf.fit(X_train, y_train)

pred = rf.predict(X_test)

print(
    "Accuracy:",
    accuracy_score(y_test, pred)
)

Accuracy: 0.8645505209595348


In [149]:
feature_importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": rf.feature_importances_
})

feature_importance.sort_values(
    by="Importance",
    ascending=False
).head(25)

,Feature,Importance
0,cycle,0.156031
25,sensor_11_avg5,0.073327
36,sensor_4_avg10,0.067465
35,sensor_11_avg10,0.064745
26,sensor_4_avg5,0.064590
12,sensor_9,0.054870
38,sensor_2_avg10,0.053454
37,sensor_15_avg10,0.052030
17,sensor_14,0.047882
27,sensor_15_avg5,0.045662


In [150]:
[col for col in X_train.columns if "avg10" in col or "std" in col]

['sensor_11_avg10',
 'sensor_4_avg10',
 'sensor_15_avg10',
 'sensor_2_avg10',
 'sensor_17_avg10']

In [151]:
important_sensors = [
    "sensor_11",
    "sensor_4",
    "sensor_15",
    "sensor_2",
    "sensor_17"
]

for sensor in important_sensors:
    df2[f"{sensor}_avg20"] = (
        df2.groupby("engine_id")[sensor]
        .transform(
            lambda x: x.rolling(
                20,
                min_periods=1
            ).mean()
        )
    )

In [152]:
X = df2.drop(
    ["health_status"],
    axis=1
)

X = X.drop(
    ["engine_id", "RUL"],
    axis=1
)

y = df2["health_status"]

In [153]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [154]:
[col for col in X_train.columns if "avg20" in col]

['sensor_11_avg20',
 'sensor_4_avg20',
 'sensor_15_avg20',
 'sensor_2_avg20',
 'sensor_17_avg20']

In [155]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=20,
    min_samples_split=5,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

pred = rf.predict(X_test)

print(
    "Accuracy:",
    accuracy_score(y_test, pred)
)

Accuracy: 0.8907196510782651


In [156]:
feature_importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": rf.feature_importances_
})

feature_importance.sort_values(
    by="Importance",
    ascending=False
).head(20)

,Feature,Importance
0,cycle,0.150464
37,sensor_15_avg10,0.057289
36,sensor_4_avg10,0.055058
35,sensor_11_avg10,0.054050
25,sensor_11_avg5,0.054022
26,sensor_4_avg5,0.052251
12,sensor_9,0.050865
17,sensor_14,0.043909
38,sensor_2_avg10,0.041622
41,sensor_4_avg20,0.037402


In [157]:
important_sensors = [
    "sensor_11",
    "sensor_4",
    "sensor_15",
    "sensor_2",
    "sensor_17"
]

for sensor in important_sensors:
    df2[f"{sensor}_avg30"] = (
        df2.groupby("engine_id")[sensor]
        .transform(
            lambda x: x.rolling(
                30,
                min_periods=1
            ).mean()
        )
    )

In [158]:
X = df2.drop(
    ["health_status"],
    axis=1
)

X = X.drop(
    ["engine_id", "RUL"],
    axis=1
)

y = df2["health_status"]

In [159]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [160]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=20,
    min_samples_split=5,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

pred = rf.predict(X_test)

print(
    "Accuracy:",
    accuracy_score(y_test, pred)
)

Accuracy: 0.9173733947177126


In [161]:
feature_importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": rf.feature_importances_
})

feature_importance.sort_values(
    by="Importance",
    ascending=False
).head(30)

,Feature,Importance
0,cycle,0.138945
36,sensor_4_avg10,0.062662
35,sensor_11_avg10,0.060691
26,sensor_4_avg5,0.056326
25,sensor_11_avg5,0.054476
12,sensor_9,0.046025
17,sensor_14,0.044075
37,sensor_15_avg10,0.037819
41,sensor_4_avg20,0.035486
44,sensor_17_avg20,0.034463


In [162]:
#Hyperparameter Tuning

In [163]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier

param_grid = {
    "n_estimators": [300, 500, 700],
    "max_depth": [20, 30, 40, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2"]
}

rf = RandomForestClassifier(
    random_state=42,
    n_jobs=-1
)

search = RandomizedSearchCV(
    rf,
    param_grid,
    n_iter=20,
    cv=3,
    scoring="accuracy",
    verbose=2,
    random_state=42,
    n_jobs=-1
)

search.fit(X_train, y_train)

Fitting 3 folds for each of 20 candidates, totalling 60 fits


,estimator,RandomForestC...ndom_state=42)
,param_distributions,"{'max_depth': [20, 30, ...], 'max_features': ['sqrt', 'log2'], 'min_samples_leaf': [1, 2, ...], 'min_samples_split': [2, 5, ...], ...}"
,n_iter,20
,scoring,'accuracy'
,n_jobs,-1
,refit,True
,cv,3
,verbose,2
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [164]:
print(search.best_params_)
print(search.best_score_)

{'n_estimators': 300, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'max_depth': 40}
0.902569183581241


In [165]:
best_rf = search.best_estimator_

pred = best_rf.predict(X_test)

from sklearn.metrics import accuracy_score

print(
    "Final Accuracy:",
    accuracy_score(y_test, pred)
)

Final Accuracy: 0.9185849285195057


In [166]:
from sklearn.metrics import confusion_matrix

confusion_matrix(y_test, pred)

array([[ 563,    0,   57],
       [   0, 2026,   81],
       [  19,  179, 1202]])

In [167]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test,
        pred
    )
)

              precision    recall  f1-score   support

    Critical       0.97      0.91      0.94       620
     Healthy       0.92      0.96      0.94      2107
     Warning       0.90      0.86      0.88      1400

    accuracy                           0.92      4127
   macro avg       0.93      0.91      0.92      4127
weighted avg       0.92      0.92      0.92      4127



In [168]:
#Saving the model

In [169]:
import joblib

joblib.dump(
    rf,
    "../models/rf_health_classifier.pkl"
)

['../models/rf_health_classifier.pkl']

In [170]:
joblib.dump(
    X_train.columns.tolist(),
    "../models/model_features.pkl"
)

['../models/model_features.pkl']

In [171]:
joblib.dump(
    ["Critical", "Warning", "Healthy"],
    "../models/class_labels.pkl"
)

['../models/class_labels.pkl']

In [172]:
import os

print(os.listdir("../models"))

['class_labels.pkl', 'model_features.pkl', 'rf_health_classifier.pkl']


In [173]:
loaded_model = joblib.load(
    "../models/rf_health_classifier.pkl"
)

loaded_model

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [175]:
loaded_model = joblib.load(
    "../models/rf_health_classifier.pkl"
)

type(loaded_model)

sklearn.ensemble._forest.RandomForestClassifier

In [176]:
loaded_model

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [177]:
hasattr(loaded_model, "classes_")

False

In [178]:
hasattr(rf, "classes_")

False

In [179]:
accuracy_score(y_test, rf.predict(X_test))

NotFittedError: This RandomForestClassifier instance is not fitted yet. Call 'fit' with appropriate arguments before using this estimator.

In [180]:
hasattr(best_rf, "classes_")

True

In [181]:
accuracy_score(y_test, best_rf.predict(X_test))

0.9185849285195057

In [183]:
import joblib

joblib.dump(
    best_rf,
    "../models/rf_health_classifier.pkl"
)

['../models/rf_health_classifier.pkl']

In [184]:
loaded_model = joblib.load(
    "../models/rf_health_classifier.pkl"
)

hasattr(loaded_model, "classes_")

True

In [185]:
sample = X_test.iloc[[0]]

loaded_model.predict(sample)

array(['Warning'], dtype=object)

In [186]:
joblib.dump(
    X_train.columns.tolist(),
    "../models/model_features.pkl"
)

['../models/model_features.pkl']

In [187]:
joblib.dump(
    ["Critical", "Warning", "Healthy"],
    "../models/class_labels.pkl"
)

['../models/class_labels.pkl']

In [189]:
import joblib

loaded_model = joblib.load(
    "../models/rf_health_classifier.pkl"
)

In [190]:
pred = loaded_model.predict(X_test)

In [191]:
from sklearn.metrics import accuracy_score

print(
    accuracy_score(y_test, pred)
)

0.9185849285195057


In [192]:
print("Best RF:", accuracy_score(y_test, best_rf.predict(X_test)))
print("Loaded :", accuracy_score(y_test, loaded_model.predict(X_test)))

Best RF: 0.9185849285195057
Loaded : 0.9185849285195057


In [188]:
from sklearn.metrics import classification_report, confusion_matrix

pred = loaded_model.predict(X_test)

print(classification_report(y_test, pred))
print(confusion_matrix(y_test, pred))

              precision    recall  f1-score   support

    Critical       0.97      0.91      0.94       620
     Healthy       0.92      0.96      0.94      2107
     Warning       0.90      0.86      0.88      1400

    accuracy                           0.92      4127
   macro avg       0.93      0.91      0.92      4127
weighted avg       0.92      0.92      0.92      4127

[[ 563    0   57]
 [   0 2026   81]
 [  19  179 1202]]


In [195]:
#Making a human readable copy into a text file
print(X_train.columns.tolist())

['cycle', 'op_setting_1', 'op_setting_2', 'op_setting_3', 'sensor_1', 'sensor_2', 'sensor_3', 'sensor_4', 'sensor_5', 'sensor_6', 'sensor_7', 'sensor_8', 'sensor_9', 'sensor_10', 'sensor_11', 'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15', 'sensor_16', 'sensor_17', 'sensor_18', 'sensor_19', 'sensor_20', 'sensor_21', 'sensor_11_avg5', 'sensor_4_avg5', 'sensor_15_avg5', 'sensor_2_avg5', 'sensor_17_avg5', 'sensor_11_diff', 'sensor_4_diff', 'sensor_15_diff', 'sensor_2_diff', 'sensor_17_diff', 'sensor_11_avg10', 'sensor_4_avg10', 'sensor_15_avg10', 'sensor_2_avg10', 'sensor_17_avg10', 'sensor_11_avg20', 'sensor_4_avg20', 'sensor_15_avg20', 'sensor_2_avg20', 'sensor_17_avg20', 'sensor_11_avg30', 'sensor_4_avg30', 'sensor_15_avg30', 'sensor_2_avg30', 'sensor_17_avg30']


In [196]:
with open("../models/features.txt", "w") as f:
    for col in X_train.columns:
        f.write(col + "\n")